In [1]:
# Installing autogluo using this set https://auto.gluon.ai/stable/install.html 
# using the pip option with gpu 

In [2]:
import sys
import os
import pandas as pd
from datetime import datetime
from sklearn.model_selection import train_test_split
from autogluon.tabular import TabularDataset, TabularPredictor

/home/grace.bcolverd/miniconda3/envs/fdl-biomass-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
def transform(df, label ):
    ignore_cols = ['heading' ]
    cols = [x for x in df.columns.tolist() if x not in ignore_cols] 
            
    working_cols = cols
    df = df[working_cols]
    df = df[~df[label].isna()]
    return df

def save_results(results, output_path):
    res_string = str(results)
    # summary = predictor.fit_summary()
    with open(os.path.join(output_path, 'model_summary.txt'), 'w') as f:
        f.write(res_string)

In [11]:
output_path= '/home/grace.bcolverd/local/models'
df = pd.read_csv('ml_ready_data_lidar_pcube_deramped.csv')


polarimetry = 1
work_df= df[df['polarimetry']==polarimetry].copy()
train_data, test_data = train_test_split(work_df, test_size=0.2, random_state=42)

train_subset_prop = 0.1

if train_subset_prop != 1:
    train_subset, _ = train_test_split(train_data, test_size=1-train_subset_prop, random_state=42)


train_subset= transform(train_subset, 'lidar')
train_subset 

size_train = len(train_subset) 


time_limit= 1000
model_preset= 'medium_quality'
label = 'lidar'
excl_models=[]
dataset_name = 'ml_ready_data_lidar_pcube_deramped2'
col_setting = 'with_x'


output_directory = f"{output_path}/dname_{dataset_name}__target_{label}__timelim_{time_limit}__modelset_{model_preset}___tsp_{train_subset_prop}__polarim_{polarimetry}__colset_{col_setting}"


predictor = TabularPredictor(label, path=output_directory).fit(train_subset, 
                                                            time_limit=time_limit,
                                                            presets=model_preset,
                                                            excluded_model_types=excl_models
                                                            )

test_data = transform(test_data, label)

# test_data.to_csv(os.path.join(output_directory, 'test_data.csv'), index=False)
y_pred = predictor.predict(test_data.drop(columns=[label]))
results = predictor.evaluate_predictions(y_true=test_data[label], y_pred=y_pred, auxiliary_metrics=True)
size_test = len(test_data)



print(results)

sizett = {'len_train' :size_train, 'len_test':size_test  }
results.update(sizett)

save_results(results, output_directory)
res = predictor.leaderboard(test_data)
res.to_csv(os.path.join(output_directory, 'leaderboard_results.csv'))

pred = predictor.feature_importance(test_data)
pred.to_csv(os.path.join(output_directory, 'feature_importance.csv'))



In [6]:
# s this the corect indexing f polar
# {'HH': 0, 'HV': 1, 'VV': 2}

In [7]:
train_data.shape, test_data.shape

((216488, 41), (54123, 41))

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.1.1
Python Version:     3.10.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Debian 6.1.94-1 (2024-06-21)
CPU Count:          8
Memory Avail:       59.05 GB / 62.81 GB (94.0%)
Disk Space Avail:   2645.13 GB / 2952.67 GB (89.6%)
Presets specified: ['medium_quality']
Beginning AutoGluon training ... Time limit = 1000s
AutoGluon will save models to "/home/grace.bcolverd/local/models/dname_ml_ready_data_lidar_pcube_deramped2__target_lidar__timelim_1000__modelset_medium_quality___tsp_0.1__polarim_1__colset_with_x"
Train Data Rows:    21648
Train Data Columns: 39
Label Column:       lidar
AutoGluon infers your prediction problem is: 'regression' (because dtype of label-column == float and many unique label-values observed).
	Label info (max, min, mean, stddev): (41.71308503069197, 0.0, 15.72395, 9.60588)
	If 'regression' is not the corre

[1000]	valid_set's rmse: 4.59642
[2000]	valid_set's rmse: 4.55302
[3000]	valid_set's rmse: 4.54284
[4000]	valid_set's rmse: 4.54427


	-4.538	 = Validation score   (-root_mean_squared_error)
	11.38s	 = Training   runtime
	0.28s	 = Validation runtime
Fitting model: LightGBM ... Training model for up to 987.56s of the 987.56s of remaining time.


[1000]	valid_set's rmse: 4.372


	-4.3565	 = Validation score   (-root_mean_squared_error)
	6.68s	 = Training   runtime
	0.09s	 = Validation runtime
Fitting model: RandomForestMSE ... Training model for up to 980.7s of the 980.69s of remaining time.
	-4.7278	 = Validation score   (-root_mean_squared_error)
	71.49s	 = Training   runtime
	0.16s	 = Validation runtime
Fitting model: CatBoost ... Training model for up to 908.1s of the 908.1s of remaining time.
	-4.3936	 = Validation score   (-root_mean_squared_error)
	180.79s	 = Training   runtime
	0.01s	 = Validation runtime
Fitting model: ExtraTreesMSE ... Training model for up to 727.26s of the 727.26s of remaining time.
	-4.7122	 = Validation score   (-root_mean_squared_error)
	9.46s	 = Training   runtime
	0.2s	 = Validation runtime
Fitting model: NeuralNetFastAI ... Training model for up to 716.54s of the 716.53s of remaining time.
	-4.8475	 = Validation score   (-root_mean_squared_error)
	30.01s	 = Training   runtime
	0.05s	 = Validation runtime
Fitting model: XGBoos

[1000]	valid_set's rmse: 4.2612


	-4.2603	 = Validation score   (-root_mean_squared_error)
	14.63s	 = Training   runtime
	0.11s	 = Validation runtime
Fitting model: WeightedEnsemble_L2 ... Training model for up to 360.0s of the 583.97s of remaining time.
	Ensemble Weights: {'KNeighborsDist': 0.5, 'LightGBMLarge': 0.375, 'LightGBM': 0.042, 'CatBoost': 0.042, 'XGBoost': 0.042}
	-3.9986	 = Validation score   (-root_mean_squared_error)
	0.02s	 = Training   runtime
	0.0s	 = Validation runtime
AutoGluon training complete, total runtime = 416.11s ... Best model: WeightedEnsemble_L2 | Estimated inference throughput: 6292.5 rows/s (2165 batch size)
TabularPredictor saved. To load, use: predictor = TabularPredictor.load("/home/grace.bcolverd/local/models/dname_ml_ready_data_lidar_pcube_deramped2__target_lidar__timelim_1000__modelset_medium_quality___tsp_0.1__polarim_1__colset_with_x")


{'root_mean_squared_error': -3.9914140653623216, 'mean_squared_error': -15.931386241172174, 'mean_absolute_error': -2.632878928904886, 'r2': 0.8297092830061146, 'pearsonr': 0.9115923571922361, 'median_absolute_error': -1.619809474400114}


These features in provided data are not utilized by the predictor and will be ignored: ['polarimetry']
Computing feature importance via permutation shuffling for 38 features using 5000 rows with 5 shuffle sets...
	190.13s	= Expected runtime (38.03s per shuffle set)
	155.73s	= Actual runtime (Completed 5 of 5 shuffle sets)
